In [1]:
import glob
import pandas as pd
import h2o
from h2o.estimators import H2OExtendedIsolationForestEstimator, H2ORandomForestEstimator
import mlflow.h2o
import mlflow

import config

In [2]:
# define Mlflow env. settings
mlflow.set_tracking_uri(uri=config.mlflow_uri)
%env MLFLOW_TRACKING_URI= config.mlflow_db
%env mlflow_tracking_username = config.mlflow_un
%env mlflow_tracking_password = config.mlflow_pw

# Initialize H2O server
h2o.init(ip="127.0.0.1", port="8080", nthreads=-1, min_mem_size= config.min_memory, max_mem_size= config.max_memory, enable_assertions=False)

# Clear h2o memory
h2o.remove_all()

env: MLFLOW_TRACKING_URI=config.mlflow_db
env: mlflow_tracking_username=config.mlflow_un
env: mlflow_tracking_password=config.mlflow_pw
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 25.411-b09, mixed mode)
  Starting server from C:\DSE\Python\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\RANDLE~1.KNI\AppData\Local\Temp\4\tmp5lenwchg
  JVM stdout: C:\Users\RANDLE~1.KNI\AppData\Local\Temp\4\tmp5lenwchg\h2o_randle_knight_started_from_python.out
  JVM stderr: C:\Users\RANDLE~1.KNI\AppData\Local\Temp\4\tmp5lenwchg\h2o_randle_knight_started_from_python.err


H2OServerError: Server process terminated with error code 1.

In [ ]:
#mlflow.autolog(log_models=True, log_datasets=True)
mlflow.set_experiment("Extended Isolation Forest Experiment")
customerList = ["X5"]#,"X3","X4","X1","X2","X6"] #,"X7","X8"
files = glob.glob(f'C:/MLAR_DATA/T0001/FS_base_data/FS_base_data_X*.parquet')
with mlflow.start_run(run_name=f"EIF_Control_ColdStart", nested=True) as grandparent_run:
    for customer in customerList:
        if any(customer in substring for substring in files): 
            filtered_files = [word for word in files if not word.endswith(customer+'.parquet')]
            df = pd.concat([pd.read_parquet(fp) for fp in filtered_files])
            
            h2oFrame = h2o.H2OFrame(python_obj=df)
            train, test = h2oFrame.split_frame(ratios =[0.7], seed=1234)
            features = train.drop(['T0083','T0083_SEVERITY','T_MLAR','T_MLAR_SEVERITY']).columns

            # Define model parameters
            numTrees = 100
            sampleSize = 256

            # Initialize the model
            eif_model = H2OExtendedIsolationForestEstimator(ntrees=numTrees, sample_size=sampleSize, seed=1234)
            
            # Start an MLflow run
            with mlflow.start_run(run_name=f"Predicted_Dataset", nested=True) as parent_run:
                # Log model parameters and train
                mlflow.log_param("ntrees", numTrees)
                mlflow.log_param("sample_size", sampleSize)
                eif_model.train(x=features,training_frame=train) 
                
                # Predict anomalies
                anomaliesTrain = eif_model.predict(train)
                anomaliesTest = eif_model.predict(test)
                
                # Combine the scores with the original data
                predictedTrainDF = train.cbind(anomaliesTrain).as_data_frame(use_pandas=True)
                predictedTestDF = test.cbind(anomaliesTest).as_data_frame(use_pandas=True)
                #predictedTrainDF["anomaly"] = (predictedTrainDF["T0083_SEVERITY"] > 1).ifelse(1, 0)
                #predictedTestDF["anomaly"] = (predictedTestDF["T0083_SEVERITY"] > 1).ifelse(1, 0)

                #Export Dataset
                mlflow.log_table(predictedTrainDF,"predictedTrainDF.json")
                mlflow.log_table(predictedTestDF,"predictedTestDF.json")

                #Train explaination model
                DRF_model = H2ORandomForestEstimator(model_id = "global_surrogate_decision_tree.hex",ntrees=numTrees)
                predictedTrain = h2o.H2OFrame(python_obj=predictedTrainDF)
                predictedTest = h2o.H2OFrame(python_obj=predictedTestDF)
                with mlflow.start_run(run_name=f"Explain_model", nested=True) as child_run:
                    DRF_model.train(x=features, y="anomaly",training_frame=predictedTrain)
                    #explainOutput = DRF_model.explain(predictedTest)
                    #predictContribution = DRF_model.predict_contributions(predictedTest)
        else:
            print("error")

In [ ]:
predictContribution = DRF_model.predict_contributions(predictedTest)

In [5]:
## Clear h2o memory
import h2o
import config
h2o.init(ip="127.0.0.1", port="8080", nthreads=-1, min_mem_size= config.min_memory, max_mem_size= config.max_memory, enable_assertions=False)
h2o.remove_all()
h2o.cluster().shutdown()

Checking whether there is an H2O instance running at http://127.0.0.1:8080..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 25.411-b09, mixed mode)
  Starting server from C:\DSE\Python\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\RANDLE~1.KNI\AppData\Local\Temp\3\tmpjw_01ina
  JVM stdout: C:\Users\RANDLE~1.KNI\AppData\Local\Temp\3\tmpjw_01ina\h2o_randle_knight_started_from_python.out
  JVM stderr: C:\Users\RANDLE~1.KNI\AppData\Local\Temp\3\tmpjw_01ina\h2o_randle_knight_started_from_python.err
  Server is running at http://127.0.0.1:8080
Connecting to H2O server at http://127.0.0.1:8080 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.3
H2O_cluster_version_age:,28 days and 25 minutes
H2O_cluster_name:,H2O_from_python_randle_knight_sd9b9x
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,112.3 Gb
H2O_cluster_total_cores:,16
H2O_cluster_allowed_cores:,16
H2O_cluster_status:,"locked, healthy"


H2O session _sid_8c0b closed.
